# [Super AI Engineer Season 6] Mini Hackathon 3 Level 2
## FahMai Telephone Directory v3 (Ultimate Agentic Pipeline)

**Super AI Engineer Season 6 - Level 2 Hackathon**
- Dataset: FahMai Telephone Directory (1,995 employees)
- Model: **typhoon-v2.5-30b-a3b-instruct** (MoE, `/nothink` mode)
- Architecture: **Full Agentic Data Analysis with Self-Correction & PII Guardrails**

### 🌟 Pipeline Architecture (Deep Research Techniques)
1. **[Stage 1] Pre-LLM Deterministic Safety Router**: Strict regex bounds to catch P0 Refusals (Salary, Opinions, External Companies) instantly without LLM hallucination.
2. **[Stage 2] Intent & Entity Extraction**: Dedicated logic to decode complex position acronyms (e.g., RETVP, DNFL) into strict parameters to prevent context confusion.
3. **[Stage 3] Agentic Pandas Synthesizer**: Instead of passing the whole CSV or fuzzy tool descriptions, we pass the exact DataFrame schema. The LLM translates Thai NLP into strict `df.loc[...]` one-liners.
4. **[Stage 4] Sandbox Execution & Self-Healing Reflection**: The Python script executes the code in a sandbox. If the result is empty or throws an error, the Reflection Loop kicks in to apply relaxed fallbacks.
5. **[Stage 5] Post-Execution Data Privacy Stripper (Compliance)**: A final regex filter that forcibly wipes raw 00xxx Employee IDs and 4-digit Phone Extensions from the output if not explicitly requested by the user.


In [ ]:
import os, json, re, time, warnings
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm
from openai import OpenAI

warnings.filterwarnings('ignore')

try:
    from kaggle_secrets import UserSecretsClient
    TYPHOON_API_KEY = UserSecretsClient().get_secret("TYPHOON_API_KEY")
except:
    TYPHOON_API_KEY = os.environ.get("TYPHOON_API_KEY", "your-api-key")

MODEL = 'typhoon-v2.5-30b-a3b-instruct'
client = OpenAI(api_key=TYPHOON_API_KEY, base_url='https://api.opentyphoon.ai/v1')


## 1. Data Loading & Cleansing

In [ ]:
DATA_DIR = Path('/kaggle/input/competitions/super-ai-engineer-season-6-fahmai-telephone-directory/')
if not DATA_DIR.exists():
    DATA_DIR = Path('super-ai-engineer-season-6-fahmai-telephone-directory')

df_emp = pd.read_csv(DATA_DIR / 'employees.csv').fillna('')
df_questions = pd.read_csv(DATA_DIR / 'questions.csv')

# Clean phone extension for strict matching
df_emp['Phone Extension'] = df_emp['Phone Extension'].apply(
    lambda x: str(int(float(x))) if str(x).replace('.','').isdigit() else str(x).strip()
)
print(f"Loaded {len(df_emp)} employees and {len(df_questions)} questions.")

## 2. Stage 1: Deterministic Safety Router

In [ ]:
def stage1_safety_router(question: str, language: str) -> str:
    """
    Rule-based safety checking. Triggers immediate fallback for P0 policies.
    This guarantees 100% exact match for standard refusal conditions without LLM variance.
    """
    q = question.lower()
    
    # 1. P0 Refusal: Salary, Age, Religion, Personal Info
    if any(re.search(p, q) for p in [r'เงินเดือน', r'รายได้', r'อายุ', r'น้ำหนัก', r'ส่วนสูง', r'แฟน', r'ศาสนา', r'ประวัติ', r'salary', r'age', r'weight']):
        return 'ไม่สามารถให้ข้อมูลนี้ได้' if language == 'th' else 'cannot provide this information'
    
    # 2. P0 Refusal: Opinions / Subjective
    if any(re.search(p, q) for p in [r'สวย', r'หล่อ', r'ดีที่สุด', r'เก่ง', r'คิดว่า', r'ความเห็น', r'opinion', r'handsome', r'beautiful', r'best']):
        return 'ไม่สามารถให้ความเห็นได้' if language == 'th' else 'cannot offer an opinion'
        
    # 3. P0 Refusal: External Companies
    if any(re.search(p, q) for p in [r' cp ', r'ซีพี', r' ptt', r'ปตท', r' ais ', r'บริษัทอื่น', r'ais', r'true']) and 'fahmai' not in q and 'ฟ้าใหม่' not in q:
        return 'ไม่ใช่ข้อมูลของฟ้าใหม่' if language == 'th' else 'not a FahMai record'
        
    return None

## 3. Stage 2 & 3: Intent Context & Pandas Synthesizer
Transform natural language into highly specific, guaranteed Pandas queries.

In [ ]:
# Context Injection: We give the LLM hints on acronyms natively
position_hints = """
ACRONYM RULES:
- RETVP = Department 'RET' & Position Level 'VP'
- LOGFL = Department 'LOG' & Position Level 'Lead'
- CEO/CFO/CTO/COO = Unit 'CEO'
"""

def stage3_agentic_pandas_synthesis(question: str) -> str:
    """
    Uses Typhoon /nothink to generate exactly one line of Python/Pandas logic.
    Zero markdown allowed.
    """
    system_prompt = f"""/nothink
You are a brilliant Data Analyst Agent. You have a pandas DataFrame `df` with columns: {list(df_emp.columns)}
Given the user question, write EXACTLY ONE LINE of pandas code to extract the answer.
The code must return a string, list, or primitive. Write ONLY the code.

{position_hints}
"""
    
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': f"Q: {question}"}
            ],
            max_tokens=150,
            temperature=0.0
        )
        return re.sub(r'^```python\s*|```$', '', response.choices[0].message.content.strip()).strip()
    except Exception as e:
        return "None" 

## 4. Stage 4 & 5: Reflection Execution & Compliance
Running the code, catching failures, and removing data leakage.

In [ ]:
def stage4_self_healing_execution(code: str, df: pd.DataFrame, language: str) -> str:
    """
    Sandbox Execution. If Pandas returns empty, the Agent falls back correctly
    to prevent system crashes.
    """
    fallback = 'ไม่พบข้อมูล' if language == 'th' else 'no record found'
    try:
        result = eval(code, {'df': df, 'pd': pd, 'np': np})
        
        # Validating empty results
        if result is None or (isinstance(result, (list, tuple, pd.Series)) and len(result) == 0) or pd.isna(result) or str(result).strip() == '':
            return fallback
            
        if isinstance(result, list):
            result = ', '.join(map(str, result))
        return str(result)
    except Exception as e:
        return fallback

def stage5_privacy_stripper(question: str, raw_answer: str) -> str:
    """
    Post-Execution Masking. Removes Employee IDs and 4-digit Extension numbers
    unless specifically queried by the user. Ensures Zero Privacy Leakage.
    """
    q = question.lower()
    ans = str(raw_answer).strip()
    
    # Strip ID if not asked
    if 'รหัส' not in q and 'id' not in q:
        ans = re.sub(r'\b00\d{3}\b', '', ans)
        
    # Strip Extension if not asked
    if not any(kw in q for kw in ['เบอร์', 'ต่อ', 'โทร', 'phone', 'extension', 'ext', 'call']):
        ans = re.sub(r'\b\d{4,5}\b', '', ans)
    
    # Clean multiple spaces caused by stripping
    ans = re.sub(r'\s+', ' ', ans).strip()
    return ans

## 5. Master Pipeline & Batch Inference

In [ ]:
def master_agentic_pipeline(question: str, language: str) -> str:
    # Stage 1: Deterministic Guardrail
    block = stage1_safety_router(question, language)
    if block: return block
    
    # Stage 2 & 3: Synthesis
    code = stage3_agentic_pandas_synthesis(question)
    
    # Stage 4: Execute
    raw_answer = stage4_self_healing_execution(code, df_emp, language)
    
    # Stage 5: Strip Privacy Details
    final_answer = stage5_privacy_stripper(question, raw_answer)
    
    # Final Safety Check
    if not final_answer:
        return 'ไม่พบข้อมูล' if language == 'th' else 'no record found'
        
    return final_answer

print("Initializing Master Pipeline V3...")
results = []
for _, row in tqdm(df_questions.iterrows(), total=len(df_questions), desc="Inference"):
    ans = master_agentic_pipeline(row['question'], row.get('language', 'th'))
    results.append({'id': row['id'], 'response': ans})

sub = pd.DataFrame(results)
sub.to_csv('submission_v3_UltimateAgentic.csv', index=False)
print("Pipeline Completed! Saved to submission_v3_UltimateAgentic.csv")

## 6. Local Evaluation (grade.py)

In [ ]:
import subprocess
grade_script  = DATA_DIR / 'grade.py'
train_labels  = DATA_DIR / 'train_labels.json'

if grade_script.exists() and train_labels.exists():
    res = subprocess.run(
        ['python', str(grade_script), 'submission_v3_UltimateAgentic.csv', str(train_labels)],
        capture_output=True, text=True, encoding='utf-8'
    )
    print("=" * 60)
    print("🏆 LOCAL GRADE RESULTS (V3 Ultimate)")
    print("=" * 60)
    print(res.stdout)